# **RNN vs LSTM**

* RNN은 이전 데이터를 기억할 수 있는 능력이 있습니다.
* 하지만 계속해서 tanh()에서 나온 결과를 곱셈하여 결과를 계산하기 때문에 long-term에서는 성능이 매우 하락합니다.
* 이를 개선한 구조가 gate RNN인 LSTM 계열 모델들입니다.
* 우리는 RNN과 LSTM을 비교하고 싶습니다.

## **A. N턴전 숫자 맞추기**

### **1. 데이터 준비**

- T,B,F (100, 64, 1) 크기의 랜덤 float형 데이터 (x)를 만들어 주세요.
- 데이터는 0부터 9사이의 숫자가 됩니다.
- 정답 (y)는 0으로만 채워진 똑같은 (T,B,F) 크기의 tensor를 만들어줍니다.
- 그리고 K번째부터의 데이터는 x의 첫번째부터 크기-K번째까지의 데이터로 덮어씌워줍니다.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

T = 100     # 시퀀스 길이 (몇 턴짜리 데이터인가)
K = 20      # 몇 턴 전 숫자를 맞출 것인가
B = 64

In [2]:
# 1. 데이터 생성 함수 변경 (배치 크기 추가)
def make_batch():
    # (T, batch_size, 1) 형태로 생성
    x = torch.randint(0, 10, (T, B, 1)).float()
    y = torch.zeros_like(x)
    y[K:] = x[:-K]
    return x.to("cuda"), y.to("cuda")

In [3]:
# code here

### **2. 모델 만들기**

레이어 구성은 다음과 같습니다.

**LSTM**
- LSTM layer(1,64) -> Linear layer(64,1)
- linear layer에 들어가는 정보는 이전 데이터를 모두 넣습니다.

**RNN**
- RNN layer(1,64) -> Linear layer(64,1)
- linear layer에 들어가는 정보는 이전 데이터를 모두 넣습니다.

In [4]:
# code here
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1,64)
        self.linear = nn.Linear(64,1)

    def forward(self, X):
        h, _ = self.rnn(X)
        return self.linear(h)

In [5]:
# code here
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1,64)
        self.linear = nn.Linear(64,1)

    def forward(self, X):
        h, _ = self.lstm(X)
        return self.linear(h)

### **3. 손실함수와 옵티마이저**

아래와 같이 옵티마이저를 설정합니다.

- Optimizer : Adam
- learning rate : 0.001
- 손실 함수는 forward 내에서 F.mse_loss를 활용합니다.

각각의 모델을 위해 2개의 별도의 optimizer를 생성해주세요.

In [6]:
# code here
lstm = LSTM()
rnn = RNN()

lstm.to('cuda')
rnn.to('cuda')

RNN(
  (rnn): RNN(1, 64)
  (linear): Linear(in_features=64, out_features=1, bias=True)
)

### **4. 학습하기**

LSTM과 RNN을 다음 정보대로 학습시켜주세요.
- epochs : 3000
- 100번의 epoch마다 loss를 출력해보도록 합시다.


In [7]:
# code here
optimizer_rnn = torch.optim.Adam(rnn.parameters(), lr = 0.001)
optimizer_lstm = torch.optim.Adam(lstm.parameters(), lr = 0.001)

In [8]:
criterion = nn.MSELoss()

### **5. 평가하기**

총 20번 새로운 데이터를 만들어 예측을 시키고, 결과를 구해 봅시다.


In [9]:
# code here
epochs = 5000

for epoch in range(epochs):
    X, y = make_batch()
    # rnn학습
    rnn.train()
    y_pred_rnn = rnn(X)
    rnn_loss = criterion(y_pred_rnn[K:], y[K:])
    optimizer_rnn.zero_grad()
    rnn_loss.backward()
    optimizer_rnn.step()


    # lstm학습
    lstm.train()
    y_pred_lstm = lstm(X)
    lstm_loss = criterion(y_pred_lstm[K:], y[K:])
    optimizer_lstm.zero_grad()
    lstm_loss.backward()
    optimizer_lstm.step()

    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1} | RNN loss: {rnn_loss} | LSTM loss: {lstm_loss}")


epoch 100 | RNN loss: 8.025225639343262 | LSTM loss: 8.023715019226074
epoch 200 | RNN loss: 8.357951164245605 | LSTM loss: 8.357111930847168
epoch 300 | RNN loss: 8.187637329101562 | LSTM loss: 8.184638977050781
epoch 400 | RNN loss: 8.148820877075195 | LSTM loss: 8.149198532104492
epoch 500 | RNN loss: 8.121214866638184 | LSTM loss: 8.121026039123535
epoch 600 | RNN loss: 8.207388877868652 | LSTM loss: 8.20756721496582
epoch 700 | RNN loss: 8.326358795166016 | LSTM loss: 8.327563285827637
epoch 800 | RNN loss: 8.179149627685547 | LSTM loss: 8.18009090423584
epoch 900 | RNN loss: 8.381148338317871 | LSTM loss: 8.380864143371582
epoch 1000 | RNN loss: 8.050236701965332 | LSTM loss: 8.050065040588379
epoch 1100 | RNN loss: 8.284713745117188 | LSTM loss: 8.2848482131958
epoch 1200 | RNN loss: 8.291415214538574 | LSTM loss: 8.293437004089355
epoch 1300 | RNN loss: 8.266931533813477 | LSTM loss: 8.266718864440918
epoch 1400 | RNN loss: 8.25216293334961 | LSTM loss: 8.253616333007812
epoch 

### **6. 턴수 바꿔보기**

**Q1. K=10으로 수정하고 학습하여 결과를 확인해보자.**
- 이전 모델과 가중치를 그대로 이어받아 학습을 진행한다.

In [10]:
# code here

**Q2. K=20으로 수정하고 학습하여 결과를 확인해보자.**
- 이전 모델과 가중치를 그대로 이어받아 학습을 진행한다.

In [11]:
# code here

**Q3. K=30으로 수정하고 학습하여 결과를 확인해보자.**
- 이전 모델과 가중치를 그대로 이어받아 학습을 진행한다.

In [12]:
# code here

## **B. 처음 숫자 맞추기**

### **1. 데이터 준비**

- T,B,F (10, 64, 1) 크기의 랜덤 float형 데이터 (x)를 만들어 주세요.
- 데이터는 0부터 9사이의 숫자가 됩니다.
- 정답 (y)는 x의 0번째 (맨 처음 숫자)로 정의합니다.

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

T = 5     # 시퀀스 길이 (몇 턴짜리 데이터인가)
B = 64

In [14]:
def make(T, B):
    x = torch.randint(0, 10, (T,B,1)).float()   # 0~9 랜덤 숫자 5개
    y = x[0]                                   # 정답 = 맨 처음 숫자
    return x, y

In [15]:
# code here

### **2. 모델 만들기**

레이어 구성은 다음과 같습니다.

**LSTM**
- LSTM layer(1,64) -> Linear layer(64,1)
- linear layer에 들어가는 정보는 rnn에서 나온 마지막 정보만 넣습니다.

**RNN**
- RNN layer(1,64) -> Linear layer(64,1)
- linear layer에 들어가는 정보는 rnn에서 나온 마지막 정보만 넣습니다.

### **3. 손실함수와 옵티마이저**

아래와 같이 옵티마이저를 설정합니다.

- Optimizer : Adam
- learning rate : 0.001
- 손실 함수는 forward 내에서 F.mse_loss를 활용합니다.

각각의 모델을 위해 2개의 별도의 optimizer를 생성해주세요.

In [17]:
# code here

### **4. 학습하기**

LSTM과 RNN을 다음 정보대로 학습시켜주세요.
- epochs : 1000
- 100번의 epoch마다 loss를 출력해보도록 합시다.


In [18]:
# code here

### **5. 평가하기**

총 20번 새로운 데이터를 만들어 예측을 시키고, 결과를 구해 봅시다.


In [19]:
# code here

### **6. timestep 바꾸기**

**Q1. timestep=10일때의 정확도를 평가해보자.**
- 이전 모델과 가중치를 그대로 이어받아 학습을 진행한다.

In [20]:
# code here

**Q2. timestep=30일때의 정확도를 평가해보자.**
- 이전 모델과 가중치를 그대로 이어받아 학습을 진행한다.

In [21]:
# code here

**Q3. timestep=50일때의 정확도를 평가해보자.**
- 이전 모델과 가중치를 그대로 이어받아 학습을 진행한다.

In [22]:
# code here

## **C. 신호 다음 숫자만 기억하기**

### **1. 데이터 준비**

- T,B,F (10, 64, 1) 크기의 랜덤 float형 데이터 (x)를 만들어 주세요.
- 데이터는 0부터 9사이의 숫자가 됩니다.
- 그리고 랜덤 위치에 10을 하나 추가합니다.
- y는 그 랜덤 위치 다음의 값이 됩니다.

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F

T = 10     # 시퀀스 길이 (몇 턴짜리 데이터인가)
B = 64

In [43]:
def make(T, B):
    x = torch.randint(0, 10, (T, B, 1)).float()   # 0~9 랜덤 숫자 (T, B, 1)
    pos = torch.randint(0, T - 1, (B,))           # 10을 넣을 위치: 0 ~ T-2 (다음 값이 있어야 함)
    idx = torch.arange(B)
    y = x[pos + 1, idx]                           # 정답 = 10 바로 다음 값, shape (B, 1)
    x[pos, idx] = 10.                             # 샘플마다 다른 위치를 10으로 교체
    return x, y

In [25]:
# code here

### **2. 모델 만들기**

레이어 구성은 다음과 같습니다.

**LSTM**
- LSTM layer(1,64) -> Linear layer(64,1)
- linear layer에 들어가는 정보는 rnn에서 나온 마지막 정보만 넣습니다.

**RNN**
- RNN layer(1,64) -> Linear layer(64,1)
- linear layer에 들어가는 정보는 rnn에서 나온 마지막 정보만 넣습니다.

In [ ]:
# code here
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1,64)
        self.linear = nn.Linear(64,1)

    def forward(self, X):
        h, _ = self.lstm(X)
        return self.linear(h[-1])

In [38]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1,64)
        self.linear = nn.Linear(64,1)

    def forward(self, X):
        h, _ = self.rnn(X)
        return self.linear(h)

In [39]:
lstm = LSTM()
rnn = RNN()

### **3. 손실함수와 옵티마이저**

아래와 같이 옵티마이저를 설정합니다.

- Optimizer : Adam
- learning rate : 0.001
- 손실 함수는 forward 내에서 F.mse_loss를 활용합니다.

각각의 모델을 위해 2개의 별도의 optimizer를 생성해주세요.

In [40]:
# code here
criterion = nn.MSELoss()

In [41]:
optimizer_rnn = torch.optim.Adam(rnn.parameters(), lr = 0.001)
optimizer_lstm = torch.optim.Adam(lstm.parameters(), lr = 0.001)

### **4. 학습하기**

LSTM과 RNN을 다음 정보대로 학습시켜주세요.
- epochs : 3000
- 100번의 epoch마다 loss를 출력해보도록 합시다.


In [45]:
# code here
epochs = 3000

for epoch in range(epochs):
    X, y = make(T, B)
    # rnn학습
    rnn.train()
    y_pred_rnn = rnn(X)
    rnn_loss = criterion(y_pred_rnn, y)
    optimizer_rnn.zero_grad()
    rnn_loss.backward()
    optimizer_rnn.step()


    # lstm학습
    lstm.train()
    y_pred_lstm = lstm(X)
    lstm_loss = criterion(y_pred_lstm, y)
    optimizer_lstm.zero_grad()
    lstm_loss.backward()
    optimizer_lstm.step()

    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1} | RNN loss: {rnn_loss} | LSTM loss: {lstm_loss}")



c:\Users\user\AppData\Local\Programs\Python\Python315\Lib\site-packages\torch\nn\modules\loss.py:630: UserWarning: Using a target size (torch.Size([64, 1])) that is different to the input size (torch.Size([10, 64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


epoch 100 | RNN loss: 8.57188606262207 | LSTM loss: 9.513734817504883
epoch 200 | RNN loss: 8.599529266357422 | LSTM loss: 9.141847610473633
epoch 300 | RNN loss: 7.281019687652588 | LSTM loss: 7.117335319519043
epoch 400 | RNN loss: 7.793267726898193 | LSTM loss: 7.677343845367432
epoch 500 | RNN loss: 8.01017951965332 | LSTM loss: 7.91976261138916
epoch 600 | RNN loss: 8.259162902832031 | LSTM loss: 7.818536281585693
epoch 700 | RNN loss: 7.615258693695068 | LSTM loss: 7.422825813293457
epoch 800 | RNN loss: 7.940720558166504 | LSTM loss: 7.550347805023193
epoch 900 | RNN loss: 7.5544633865356445 | LSTM loss: 7.206082820892334
epoch 1000 | RNN loss: 6.354546546936035 | LSTM loss: 5.989132881164551
epoch 1100 | RNN loss: 7.7153120040893555 | LSTM loss: 7.115124702453613
epoch 1200 | RNN loss: 5.000531196594238 | LSTM loss: 5.658565521240234
epoch 1300 | RNN loss: 5.094226360321045 | LSTM loss: 4.567558765411377
epoch 1400 | RNN loss: 5.632447242736816 | LSTM loss: 4.86480188369751
epo

### **5. 평가하기**

총 20번 새로운 데이터를 만들어 예측을 시키고, 결과를 구해 봅시다.


In [29]:
# code here

### **6. timestep 바꾸기**

**Q1. timestep=20일때의 정확도를 평가해보자.**
- 이전 모델과 가중치를 그대로 이어받아 학습을 진행한다.

In [30]:
# code here

**Q2. timestep=30일때의 정확도를 평가해보자.**
- 이전 모델과 가중치를 그대로 이어받아 학습을 진행한다.

In [31]:
# code here

**Q3. timestep=50일때의 정확도를 평가해보자.**
- 이전 모델과 가중치를 그대로 이어받아 학습을 진행한다.

In [32]:
# code here